# Assistente Clínico — Demonstração (Tech Challenge Fase 3)

Notebook 03 do projeto. Executa o assistente de ponta a ponta com o modelo
fine-tuned, sobre os 8 pacientes fictícios da base de prontuários.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente → GPU **T4** (ou L4).

## O que este notebook demonstra

| Requisito do desafio | Onde aparece |
|---|---|
| Pipeline integrando a LLM customizada | Seções 4 e 5 |
| Consulta a base de dados estruturada | Seção 6 — prontuários |
| Contextualização com dados do paciente | Seção 7 — respostas |
| Fluxo de decisão automatizado | Seção 3 — diagrama do grafo |
| Limites de atuação (nunca prescrever sem validação) | Seção 8 — guardrails |
| Logging para auditoria | Seção 9 — registros JSONL |
| Explainability (fonte da informação) | Seção 7 — citação de protocolos |

## Arquitetura

O código não vive neste notebook. Ele é importado de `src/`, no repositório —
o requisito 4 pede projeto modularizado em Python. Aqui só se orquestra a
demonstração.

```
START → carregar_paciente → recuperar_protocolos → consultar_modelo
      → decidir_desfecho → [verificar_exames | sugerir_conduta | emitir_alerta]
      → finalizar → END
```

Duas decisões de projeto que a demonstração evidencia:

**O roteamento é determinístico, não delegado ao LLM.** Em teste anterior, o
modelo falhou em emitir um rótulo de decisão válido em 2 de 2 casos. A decisão
passou para código, que lê exames pendentes e sinais vitais do prontuário. O
rótulo do modelo continua registrado, como métrica de concordância.

**A recuperação usa o escopo curado do prontuário.** Filtrar por similaridade
de texto não separava protocolo pertinente de irrelevante (medianas de 0.558 e
0.550 em cosseno). O campo `protocolos_relacionados` de cada paciente é sinal
mais confiável. A mudança levou a recuperação de 3/11 para 11/11 acertos.


## 1. Ambiente

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "SEM GPU — a demonstração roda, mas lenta"


In [ ]:
GH_USER = "mvaraujo1977"
GH_REPO = "TECH-CHALLENGE-3"

!rm -rf /content/{GH_REPO}
!git clone -q https://github.com/{GH_USER}/{GH_REPO}.git /content/{GH_REPO}

%cd /content/{GH_REPO}
!ls


### Dependências

`bitsandbytes` habilita a quantização 4-bit, que reduz o modelo de ~12 GB para
~2,3 GB — é o que permite rodar na T4 com folga.

In [ ]:
!pip install -q -U langchain langchain-community langchain-huggingface langchain-chroma langgraph chromadb sentence-transformers peft bitsandbytes accelerate

import torch
from importlib.metadata import version
for pacote in ("transformers", "peft", "langchain-core", "langgraph", "chromadb"):
    try:
        print(f"{pacote:18s} {version(pacote)}")
    except Exception:
        print(f"{pacote:18s} (ausente)")
print(f"{'torch':18s} {torch.__version__} | CUDA: {torch.cuda.is_available()}")


### Verificação dos módulos

O repositório traz um script que valida cada camada isoladamente, sem baixar
modelo. Se algo estiver quebrado, aparece aqui em segundos em vez de depois de
minutos de download.

In [ ]:
!python verificar_ambiente.py


## 2. Configuração

Os parâmetros vivem em `src/config.py`. Vale conhecê-los antes de interpretar
os resultados.

In [ ]:
import sys
sys.path.insert(0, "/content/" + GH_REPO)

from src import config

print("--- Modelo ---")
print("base    :", config.MODELO_BASE)
print("adapter :", config.ADAPTER_LORA)
print()
print("--- RAG ---")
print("embeddings :", config.MODELO_EMBEDDINGS)
print("top_k      :", config.TOP_K)
print("piso relev.:", config.LIMITE_RELEVANCIA, "(só para busca livre)")
print()
print("--- Decisão ---")
print("rótulos :", config.ROTULOS_DESFECHO)
print("fallback:", config.DESFECHO_PADRAO)


## 3. O grafo de decisão

Diagrama gerado a partir do grafo compilado — não é um desenho à parte que pode
divergir do código.

In [ ]:
from src.graph.fluxo import construir_grafo


class RetrieverVazio:
    """Dublê só para compilar o grafo e extrair o diagrama."""

    def invoke(self, consulta, codigos=None):
        return []


grafo_diagrama = construir_grafo(RetrieverVazio(), lambda p, contexto="": "")

try:
    from IPython.display import Image, display
    display(Image(grafo_diagrama.get_graph().draw_mermaid_png()))
except Exception as erro:
    print(f"(render indisponível: {erro})\n")
    print(grafo_diagrama.get_graph().draw_mermaid())


## 4. Base de conhecimento (RAG)

Os 14 protocolos em `data/protocolos/` são fatiados e indexados no Chroma. O
frontmatter YAML de cada arquivo (`codigo`, `titulo`, `versao`, `setor`) vira
metadado dos chunks — é o que permite citar a fonte exata na resposta.

In [ ]:
from src.rag.documentos import carregar_protocolos, dividir_em_chunks, resumir_carga

documentos = carregar_protocolos()
chunks = dividir_em_chunks(documentos)

print(resumir_carga(chunks))


In [ ]:
from src.rag.vectorstore import criar_embeddings, criar_retriever, indexar

# Primeira execução baixa o bge-m3 (~2 GB) e calcula os embeddings.
embeddings = criar_embeddings()
vectorstore = indexar(embeddings=embeddings, recriar=True)
retriever = criar_retriever(vectorstore)

print("Índice pronto:", vectorstore._collection.count(), "chunks")


### Recuperação com escopo curado

Comparação entre a busca livre (só similaridade) e a busca restrita aos
protocolos que o prontuário indica. É a diferença que levou a recuperação de
3/11 para 11/11 acertos.

In [ ]:
from src.rag import prontuarios as pr

paciente = pr.buscar_paciente("PAC-007")
consulta = paciente["admissao"]["queixa"]
escopo = paciente["protocolos_relacionados"]

print(f"Paciente: {paciente['id']} — {consulta}")
print(f"Protocolos curados no prontuário: {escopo}\n")

livres = retriever.invoke(consulta)
print("Busca livre (só similaridade):")
for doc in livres:
    print(f"  {doc.metadata['codigo']:14s} {doc.metadata['titulo'][:45]}")

restritos = retriever.invoke(consulta, codigos=escopo)
print("\nBusca com escopo do prontuário:")
for doc in restritos:
    print(f"  {doc.metadata['codigo']:14s} {doc.metadata['titulo'][:45]}")


## 5. Modelo fine-tuned

Carrega o Qwen2.5-3B em 4-bit e aplica os adapters LoRA. O código detecta a GPU
e escolhe o caminho quantizado automaticamente.

In [ ]:
from src.llm.modelo import carregar_modelo

mc = carregar_modelo()
print(mc.descrever())


## 6. Montagem do assistente

O `AssistenteClinico` recebe o retriever e a função de geração por injeção de
dependência — o mesmo motivo pelo qual o grafo pôde ser testado sem GPU.

In [ ]:
from src.auditoria.registro import Auditoria
from src.graph.fluxo import AssistenteClinico
from src.llm.modelo import gerar as gerar_resposta


def gerar(pergunta: str, contexto: str = "") -> str:
    return gerar_resposta(mc, pergunta, contexto=contexto)


auditoria = Auditoria(nome="demo.jsonl")
assistente = AssistenteClinico(
    retriever=retriever,
    gerar=gerar,
    auditoria=auditoria,
    nome_modelo=mc.descrever(),
)

print("Assistente pronto.\n")
print("Pacientes na base:")
for p in pr.listar_pacientes():
    paciente = pr.buscar_paciente(p["id"])
    pendentes = len(pr.exames_pendentes(paciente))
    gravidade = len(pr.sinais_de_gravidade(paciente))
    print(f"  {p['id']}  {p['idade']:>2}a {p['sexo']}  "
          f"pendentes={pendentes} gravidade={gravidade}  {p['queixa'][:44]}")


## 7. Execução

Cada consulta percorre o grafo inteiro. Ajuste `PACIENTES` para rodar um
subconjunto — os 8 levam poucos minutos na T4.

In [ ]:
PACIENTES = [p["id"] for p in pr.listar_pacientes()]   # ou ["PAC-001", "PAC-008"]
PERGUNTA = "Qual a conduta indicada para este paciente?"

resultados = []

for id_paciente in PACIENTES:
    print("=" * 78)
    paciente = pr.buscar_paciente(id_paciente)
    print(f"{id_paciente} — {paciente['admissao']['queixa']}")
    print("=" * 78)

    resposta, registro = assistente.consultar(PERGUNTA, id_paciente=id_paciente)
    resultados.append(registro)

    print(resposta)
    print()
    print(f"→ {registro.resumo()}")
    print(f"→ decisão: {registro.desfecho} ({registro.motivo_desfecho})")
    print(f"→ caminho: {' → '.join(registro.caminho_no_grafo)}")
    print()


## 8. Segurança e validação

O requisito 3 pede limites de atuação, logging e explainability. As três
verificações abaixo medem se cada um foi atendido nas respostas geradas.

In [ ]:
from src.llm.modelo import tem_guardrail

total = len(resultados)

print("=" * 62)
print("VERIFICAÇÃO DE SEGURANÇA")
print("=" * 62)

com_fonte = sum(1 for r in resultados if r.fontes)
print(f"\nExplainability — resposta com fonte citada: {com_fonte}/{total}")
for r in resultados:
    codigos = ", ".join(f["codigo"] for f in r.fontes) or "(nenhuma)"
    print(f"  {r.id_paciente}: {codigos}")

guardrail_final = sum(1 for r in resultados if tem_guardrail(r.resposta))
inserido = sum(1 for r in resultados if r.guardrail_adicionado)
print(f"\nLimite de atuação — ressalva de validação na resposta final: {guardrail_final}/{total}")
print(f"  espontânea do modelo: {guardrail_final - inserido}/{total}")
print(f"  inserida por código : {inserido}/{total}")
print("\n  A inserção por código é a segunda camada: o fine-tuning ensinou o")
print("  modelo a incluir a ressalva, mas nenhum modelo é determinístico.")


### Concordância entre o LLM e a regra determinística

O modelo continua emitindo o rótulo `DESFECHO:`, mas ele não decide o
roteamento. Comparar os dois mede quão confiável seria delegar a decisão ao
LLM — e é a justificativa medida para não delegar.

In [ ]:
validos = [r for r in resultados if r.desfecho_do_modelo]
concordaram = [r for r in resultados if r.concorda_com_modelo is True]

print(f"LLM emitiu rótulo válido : {len(validos)}/{total}")
if validos:
    print(f"LLM concordou com a regra: {len(concordaram)}/{len(validos)}")

print("\nDetalhe por paciente:")
for r in resultados:
    rotulo = r.desfecho_do_modelo or "(inválido/ausente)"
    if r.concorda_com_modelo is None:
        marca = "—"
    else:
        marca = "concorda" if r.concorda_com_modelo else "DISCORDA"
    print(f"  {r.id_paciente}: regra={r.desfecho:17s} LLM={rotulo:18s} {marca}")


## 9. Auditoria

Cada consulta grava um registro em JSONL com pergunta, paciente, trechos
recuperados, desfecho, motivo, caminho no grafo e duração. Append-only: registro
de auditoria não se sobrescreve.

In [ ]:
import json

print("Arquivo:", auditoria.arquivo)
print()
print("--- Estatísticas ---")
print(json.dumps(auditoria.estatisticas(), ensure_ascii=False, indent=2))


In [ ]:
# Um registro completo, para mostrar a estrutura
registros = auditoria.ler(limite=1)
if registros:
    print(json.dumps(registros[0], ensure_ascii=False, indent=2)[:2200])


### Tabela resumo

Consolida a execução num quadro único — material direto para o relatório.

In [ ]:
cabecalho = f"{'Paciente':9s} {'Desfecho':17s} {'Fontes':22s} {'Guard.':7s} {'Tempo':>7s}"
print(cabecalho)
print("-" * len(cabecalho))

for r in resultados:
    fontes = ",".join(f["codigo"].replace("PROT-", "P") for f in r.fontes)[:21]
    guarda = "código" if r.guardrail_adicionado else "modelo"
    print(f"{r.id_paciente:9s} {r.desfecho:17s} {fontes:22s} {guarda:7s} {r.duracao_s:6.1f}s")


## 10. Consulta livre

Para demonstração ao vivo: pergunta sem paciente vinculado. Sem dados
estruturados, a decisão é sempre `SUGERIR_CONDUTA` e a resposta é informativa.

In [ ]:
resposta, registro = assistente.consultar(
    "Quais exames são obrigatórios no protocolo de pré-operatório eletivo?"
)

print(resposta)
print()
print(f"→ {registro.desfecho} ({registro.motivo_desfecho})")


## 11. Comparação com o modelo base (opcional)

Carrega o Qwen2.5-3B **sem** os adapters LoRA e compara as respostas. É a
evidência de que o fine-tuning mudou o comportamento — sem baseline, não há como
afirmar isso.

Custa alguns minutos e mais VRAM. Rode só se quiser a comparação no relatório.

In [ ]:
EXECUTAR_COMPARACAO = False   # mude para True

if EXECUTAR_COMPARACAO:
    # adapter="" carrega apenas o modelo base
    mc_base = carregar_modelo(adapter="")
    print(mc_base.descrever(), "\n")

    def gerar_base(pergunta, contexto=""):
        return gerar_resposta(mc_base, pergunta, contexto=contexto)

    assistente_base = AssistenteClinico(
        retriever=retriever,
        gerar=gerar_base,
        auditoria=Auditoria(nome="demo_base.jsonl"),
        nome_modelo=mc_base.descrever(),
    )

    for id_paciente in ["PAC-001", "PAC-008"]:
        print("=" * 78)
        print(f"{id_paciente} — MODELO BASE (sem fine-tuning)")
        print("=" * 78)
        resposta_base, reg_base = assistente_base.consultar(PERGUNTA, id_paciente)
        print(resposta_base[:900])
        print(f"\n→ rótulo do LLM: {reg_base.desfecho_do_modelo or '(nenhum)'}")
        print(f"→ guardrail espontâneo: {not reg_base.guardrail_adicionado}")
        print()
else:
    print("Comparação desativada. Mude EXECUTAR_COMPARACAO para True.")


## 12. Salvar artefatos

In [ ]:
from google.colab import files

destino = "/content/artefatos_demo"
!mkdir -p {destino}
!cp logs/*.jsonl {destino}/ 2>/dev/null

# Diagrama do grafo, para o relatório
try:
    with open(f"{destino}/diagrama_grafo.png", "wb") as f:
        f.write(grafo_diagrama.get_graph().draw_mermaid_png())
except Exception as erro:
    print(f"(diagrama não gerado: {erro})")

!ls -la {destino}
!cd /content && zip -qr artefatos_demo.zip artefatos_demo
files.download("/content/artefatos_demo.zip")


## Para o relatório técnico

**Descrição do assistente** — as três camadas e o que cada uma resolve:

| Camada | Papel | Onde vive o conhecimento |
|---|---|---|
| Modelo fine-tuned | Formato e comportamento: citar fonte, exigir validação | Adapters LoRA |
| RAG (LangChain) | Conteúdo factual dos protocolos | Vector store |
| Grafo (LangGraph) | Decisão de fluxo a partir do prontuário | Regra determinística |

**Diagrama do fluxo LangChain** — `diagrama_grafo.png`, gerado do grafo
compilado (seção 3), não desenhado à parte.

**Avaliação e análise** — três medições feitas neste notebook:
- Recuperação: acertos por paciente contra o escopo curado do prontuário
- Segurança: citação de fonte e presença de ressalva, espontânea vs. inserida
- Concordância: quantas vezes o rótulo do LLM coincidiu com a regra

**Decisões de projeto que a medição justificou:**

1. **Roteamento determinístico.** O modelo falhou em emitir rótulo válido em
   2 de 2 casos testados (`SUGERIR CONDUÇÃO` num, nenhum rótulo no outro). A
   decisão passou para código sobre dados estruturados.
2. **Escopo curado na recuperação.** Filtro por similaridade não separava
   pertinente de irrelevante (medianas 0.558 e 0.550 em cosseno). O campo
   `protocolos_relacionados` levou a recuperação de 3/11 a 11/11.
3. **Guardrail verificado no texto do modelo, isolado das ações.** Verificar o
   texto montado dava falso positivo — a ação "Acionar o médico responsável"
   casava com os termos da ressalva, e a resposta saía sem o aviso.

**Limitações a declarar:**
- 8 pacientes e 14 protocolos: demonstração, não validação estatística
- Curadoria dos `protocolos_relacionados` assumida correta; num sistema real
  seria preciso validar essa atribuição
- Protocolos, doses e códigos são sintéticos, sem revisão clínica
- A verificação de guardrail é por palavra-chave: mede presença de padrão, não
  adequação da ressalva ao conteúdo